# DVF Data Cleaner — Paris
Loads a single DVF CSV file from `/data`, cleans and filters it, then saves a consolidated output.

## ⚙️ Configuration

In [18]:
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────
INPUT_FILE  = Path("data") / "ValeursFoncieres-2025.txt"   # ← update this filename
OUTPUT_FILE = Path("dvf_paris_clean.csv")

# ── File format ────────────────────────────────────────────
SEP = "|"   # pipe-separated by default (data.gouv.fr format)

# ── Filters ────────────────────────────────────────────────
CODE_DEPARTEMENT = "75"                         # Paris
TYPES_TO_KEEP    = ["Appartement", "Maison"]    # ignore garages, deps, etc.

# ── Outlier bounds ─────────────────────────────────────────
SURFACE_MIN  =     5   # m²
SURFACE_MAX  =  1000   # m²
PRIX_M2_MIN  =  1_000  # €/m²
PRIX_M2_MAX  = 50_000  # €/m²

## 📦 Imports

In [19]:
import pandas as pd
import numpy as np

pd.set_option("display.float_format", "{:,.2f}".format)

## 📂 Step 1 — Load file

In [20]:
COLUMNS_TO_KEEP = [
    "Date mutation",
    "Nature mutation",
    "Valeur fonciere",
    "No voie",
    "Type de voie",
    "Voie",
    "Code postal",
    "Commune",
    "Code departement",
    "Code commune",
    "Section",
    "Code type local",
    "Type local",
    "Surface reelle bati",
    "Nombre pieces principales",
    "Nature culture",
]

print(f"Loading: {INPUT_FILE}")
df = pd.read_csv(INPUT_FILE, sep=SEP, dtype=str, low_memory=False)
print(f"   → {len(df):,} rows, {df.shape[1]} columns")

# Keep only relevant columns (warn if any are missing)
available = [c for c in COLUMNS_TO_KEEP if c in df.columns]
missing   = [c for c in COLUMNS_TO_KEEP if c not in df.columns]
if missing:
    print(f"Columns not found (skipped): {missing}")

df = df[available].copy()
print(f"   → {len(available)} columns selected")

Loading: data\ValeursFoncieres-2025.txt
   → 3,714,829 rows, 43 columns
   → 16 columns selected


## 🗺️ Step 2 — Filter: Paris only + Ventes + Property types

In [21]:
# Paris only
df["Code departement"] = df["Code departement"].astype(str).str.strip()
df = df[df["Code departement"] == CODE_DEPARTEMENT].copy()
print(f"After Paris filter      : {len(df):,} rows")

# Sales only
df["Nature mutation"] = df["Nature mutation"].astype(str).str.strip()
df = df[df["Nature mutation"] == "Vente"].copy()
print(f"After Vente filter      : {len(df):,} rows")

# Property type
df["Type local"] = df["Type local"].astype(str).str.strip()
df = df[df["Type local"].isin(TYPES_TO_KEEP)].copy()
print(f"After type filter       : {len(df):,} rows")

After Paris filter      : 84,440 rows
After Vente filter      : 83,447 rows
After type filter       : 37,733 rows


## 🔢 Step 3 — Parse numerics & dates

In [22]:
# Price — French comma decimal
df["Valeur fonciere"] = (
    df["Valeur fonciere"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .str.replace(" ", "", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

# Surface
df["Surface reelle bati"] = (
    df["Surface reelle bati"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

# Rooms
df["Nombre pieces principales"] = pd.to_numeric(
    df["Nombre pieces principales"], errors="coerce"
)

# Dates
df["Date mutation"] = pd.to_datetime(df["Date mutation"], dayfirst=True, errors="coerce")
df["annee"] = df["Date mutation"].dt.year
df["mois"]  = df["Date mutation"].dt.month

print("Numeric & date parsing done.")
df[["Valeur fonciere", "Surface reelle bati", "Nombre pieces principales", "annee"]].describe()

Numeric & date parsing done.


,Valeur fonciere,Surface reelle bati,Nombre pieces principales,annee
count,"37,222.00","37,731.00","37,731.00","37,733.00"
mean,"3,085,648.26",53.07,2.39,"2,025.00"
std,"19,136,337.26",41.14,1.31,0.00
min,1.00,1.00,0.00,"2,025.00"
25%,"262,000.00",27.00,1.00,"2,025.00"
50%,"450,000.00",42.00,2.00,"2,025.00"
75%,"921,562.50",67.00,3.00,"2,025.00"
max,"695,000,000.00","1,469.00",20.00,"2,025.00"


## 🏙️ Step 4 — Arrondissement & price per m²

In [23]:
# Arrondissement from last 2 digits of Code commune (75101 → 1, 75120 → 20)
df["Code commune"] = df["Code commune"].astype(str).str.strip()
df["arrondissement"] = pd.to_numeric(
    df["Code commune"].str[-2:], errors="coerce"
).astype("Int64")

# Price per m²
df["prix_m2"] = df["Valeur fonciere"] / df["Surface reelle bati"]

print("Arrondissements found:", sorted(df["arrondissement"].dropna().unique().tolist()))

Arrondissements found: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


## 🧹 Step 5 — Remove outliers

In [24]:
before = len(df)

df = df[df["Valeur fonciere"] > 0]
df = df[df["Surface reelle bati"].between(SURFACE_MIN, SURFACE_MAX)]
df = df[df["prix_m2"].between(PRIX_M2_MIN, PRIX_M2_MAX)]

print(f"Removed {before - len(df):,} outlier rows → {len(df):,} rows remain")

Removed 5,481 outlier rows → 32,252 rows remain


## ✏️ Step 6 — Rename columns

In [25]:
df = df.rename(columns={
    "Date mutation"            : "date_mutation",
    "Nature mutation"          : "nature_mutation",
    "Valeur fonciere"          : "valeur_fonciere",
    "No voie"                  : "numero_voie",
    "Type de voie"             : "type_voie",
    "Voie"                     : "nom_voie",
    "Code postal"              : "code_postal",
    "Commune"                  : "commune",
    "Code departement"         : "code_departement",
    "Code commune"             : "code_commune",
    "Section"                  : "section",
    "Code type local"          : "code_type_local",
    "Type local"               : "type_local",
    "Surface reelle bati"      : "surface_m2",
    "Nombre pieces principales": "nb_pieces",
    "Nature culture"           : "nature_culture",
})

df.head(3)

,date_mutation,nature_mutation,valeur_fonciere,numero_voie,type_voie,nom_voie,code_postal,commune,code_departement,code_commune,section,code_type_local,type_local,surface_m2,nb_pieces,nature_culture,annee,mois,arrondissement,prix_m2
3630391,2025-01-03,Vente,"430,500.00",88,RUE,MARCADET,75018,PARIS 18,75,118,BJ,2,Appartement,48.00,3.00,NaN,2025,1,18,"8,968.75"
3630392,2025-01-03,Vente,"527,700.00",161,RUE,MARCADET,75018,PARIS 18,75,118,AV,2,Appartement,40.00,1.00,NaN,2025,1,18,"13,192.50"
3630396,2025-01-06,Vente,"383,000.00",78,RUE,DE L AQUEDUC,75010,PARIS 10,75,110,AF,2,Appartement,50.00,2.00,NaN,2025,1,10,"7,660.00"


## 📊 Step 7 — Summary

In [26]:
print("=" * 45)
print("DVF Paris — Cleaned dataset summary")
print("=" * 45)
print(f"Total transactions  : {len(df):,}")
print(f"Years covered       : {df['annee'].min():.0f} → {df['annee'].max():.0f}")
print(f"Arrondissements     : {sorted(df['arrondissement'].dropna().unique().tolist())}")
print(f"Median price / m²   : {df['prix_m2'].median():,.0f} €")
print(f"Property types      :")
print(df["type_local"].value_counts().to_string())

DVF Paris — Cleaned dataset summary
Total transactions  : 32,252
Years covered       : 2025 → 2025
Arrondissements     : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Median price / m²   : 9,930 €
Property types      :
type_local
Appartement    32075
Maison           177


## 💾 Step 8 — Save output

In [27]:
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"Saved → {OUTPUT_FILE}  ({len(df):,} rows)")

Saved → dvf_paris_clean.csv  (32,252 rows)


In [28]:
grouped = df.groupby(['annee', 'mois', 'arrondissement'])

In [31]:
result = grouped.agg(
    nb_transactions=('prix_m2', 'count'),
    avg_prix_m2=('prix_m2', 'mean'),
    avg_surface=('surface_m2', 'mean')
).reset_index()

result['intensity_index'] = (
    result['nb_transactions'] / result['avg_surface']
) * result['avg_prix_m2']

result['intensity_norm'] = (
    (result['intensity_index'] - result['intensity_index'].min()) /
    (result['intensity_index'].max() - result['intensity_index'].min())
)

display(result)

,annee,mois,arrondissement,nb_transactions,avg_prix_m2,avg_surface,intensity_index,intensity_norm
0,2025,1,1,26,"13,014.29",54.50,"6,208.65",0.03
1,2025,1,2,23,"11,598.95",55.22,"4,831.37",0.02
2,2025,1,3,53,"12,308.41",44.53,"14,650.13",0.12
3,2025,1,4,51,"15,921.35",55.67,"14,586.62",0.12
4,2025,1,5,84,"15,023.22",48.95,"25,779.15",0.23
...,...,...,...,...,...,...,...,...
235,2025,12,16,238,"12,658.35",90.42,"33,318.75",0.30
236,2025,12,17,245,"11,830.10",57.47,"50,433.37",0.47
237,2025,12,18,311,"11,929.54",42.68,"86,924.61",0.84
238,2025,12,19,168,"9,471.81",49.91,"31,882.21",0.29


In [33]:
print("Top 10 arrondissements by intensity index:")
top_intensity = result.sort_values('intensity_norm', ascending=False).head(10)
print(top_intensity[['annee', 'mois', 'arrondissement', 'intensity_norm']])


Top 10 arrondissements by intensity index:
     annee  mois  arrondissement  intensity_norm
57    2025     3              18            1.00
137   2025     7              18            0.96
130   2025     7              11            0.90
134   2025     7              15            0.84
54    2025     3              15            0.84
237   2025    12              18            0.84
177   2025     9              18            0.82
197   2025    10              18            0.78
50    2025     3              11            0.70
170   2025     9              11            0.65
